# SNI-21 — pisahkan Adrian dan Faruq

Notebook ini **tidak training** dan **tidak membuka test**. A0 gabungan hanya dipulihkan untuk split train/validation, kemudian dipisahkan menjadi dua dataset development independen dengan ID kelas SNI-21 yang sama.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, subprocess, sys
from pathlib import Path

REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/add-vadcp-pipeline'
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run([
    'git', 'clone', '--depth', '1', '--branch', BRANCH,
    'https://github.com/ediprin/coffee-bean-detection.git', str(REPO),
], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
SRC = REPO / 'src'
sys.path.insert(0, str(SRC))
os.chdir(REPO)
import coffee_detector
print('IMPORT:', coffee_detector.__file__)


In [ ]:
DRIVE_ROOTS = [
    Path('/content/drive/MyDrive'),
    Path('/content/drive/.shortcut-targets-by-id'),
]
preferred = [
    Path('/content/drive/MyDrive/Coffee_Bean_Detection/bundles/sni21-vadcp-pilot-bundle/A0_real.tar'),
    Path('/content/drive/MyDrive/coffee-bean-detection/sni21-vadcp-pilot-bundle/A0_real.tar'),
]
A0_ARCHIVE = next((path for path in preferred if path.is_file()), None)
if A0_ARCHIVE is None:
    matches = []
    for root in DRIVE_ROOTS:
        if root.is_dir():
            matches.extend(path for path in root.rglob('A0_real.tar') if path.is_file())
    matches = sorted(set(matches))
    assert matches, 'A0_real.tar tidak ditemukan di MyDrive atau shortcut.'
    A0_ARCHIVE = matches[0]

PROJECT_ROOT = Path('/content/drive/MyDrive/Coffee_Bean_Detection')
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
COMBINED_ROOT = Path('/content/sni21-a0-development')
SEPARATED_ROOT = Path('/content/sni21-source-separated-v1')
REPORT_ROOT = PROJECT_ROOT / 'evidence/sni21-source-separation-v1'
REPORT_ROOT.mkdir(parents=True, exist_ok=True)
print('ARCHIVE:', A0_ARCHIVE)
print('OUTPUT :', SEPARATED_ROOT)


In [ ]:
from coffee_detector.archive_sni21_pilot import restore_real_a0_development

restore_real_a0_development(A0_ARCHIVE, COMBINED_ROOT)
assert (COMBINED_ROOT / 'train/images').is_dir()
assert (COMBINED_ROOT / 'val/images').is_dir()
assert not (COMBINED_ROOT / 'test').exists(), 'Test tidak boleh dipulihkan.'
print('A0 DEVELOPMENT SIAP; TEST TETAP TERKUNCI')


In [ ]:
import json
from coffee_detector.separate_sni21_sources import separate_sni21_sources

summary = separate_sni21_sources(COMBINED_ROOT, SEPARATED_ROOT, link_mode='auto')
assert summary['test_locked'] is True
assert summary['test_images_accessed'] is False
assert summary['training_executed'] is False

for name in ('source_separation_summary.json', 'source_separation_manifest.json'):
    shutil.copy2(SEPARATED_ROOT / name, REPORT_ROOT / name)
for source in summary['sources']:
    shutil.copy2(SEPARATED_ROOT / source / 'audit.json', REPORT_ROOT / f'{source}_audit.json')
print(json.dumps(summary, indent=2, ensure_ascii=False))
print('SAVED:', REPORT_ROOT)


In [ ]:
import pandas as pd
from IPython.display import display

table = pd.DataFrame(summary['rows'])
display(table)
assert table['audit_safe'].all(), 'Salah satu dataset belum aman.'
print('=== PUTUSAN ===')
print('Adrian dan Faruq sudah terpisah.')
print('Training dijalankan :', summary['training_executed'])
print('Test diakses        :', summary['test_images_accessed'])
print('Kirim tabel ini. Jangan training sebelum statistik kelas dinilai.')
